In [3]:
# ============================================
# FLEET MANAGEMENT SYSTEM
# ============================================

import pandas as pd

SHIFT = 12
LOAD = 3
UNLOAD = 3
QUEUE = 6
PAYLOAD = 240

# ===================================================
# DATA SET
# ===================================================
fleet = [
    {"operator":"Shuree","badge":"001","delay":0.5,"idle":0,"down":3,
     "distance":7,"speed":30,"inter":2,"wait":1},

    {"operator":"Sausan","badge":"002","delay":1,"idle":2.1,"down":1,
     "distance":6.7,"speed":23,"inter":0,"wait":1},

    {"operator":"Ummi","badge":"003","delay":3,"idle":0,"down":4,
     "distance":8,"speed":32,"inter":3,"wait":0},

    {"operator":"Rahma","badge":"004","delay":3,"idle":2.5,"down":1,
     "distance":4.5,"speed":18,"inter":0,"wait":0},

    {"operator":"Billy","badge":"005","delay":0,"idle":0,"down":12,
     "distance":0,"speed":0,"inter":0,"wait":0},
]


# ===================================================
# NORMALIZATION
# ===================================================
def fix_number(v):
    return v.replace(",", ".")

def fix_name(v):
    return " ".join([w.capitalize() for w in v.strip().split()])


# ===================================================
# CALCULATION
# ===================================================
def calc(d):
    ready = SHIFT - (d["delay"] + d["idle"] + d["down"])
    d["ready"] = max(0, ready)

    if d["ready"] <= 0:
        d.update({"cycle":0,"tph":0,"prod":0,"tphh":0,"trips":0})
        return d

    travel = (d["distance"] / d["speed"]) * 60 if d["speed"] > 0 else 0
    obs = (d["inter"] * 4) + (d["wait"] * 5)
    cycle = LOAD + travel + UNLOAD + travel + QUEUE + obs

    d["cycle"] = round(cycle, 2)
    d["tphh"] = round(60 / cycle, 2)           # trips per hour
    d["trips"] = round(d["tphh"] * d["ready"], 2)
    d["tph"] = round(d["tphh"] * PAYLOAD, 2)   # tons per hour
    d["prod"] = round(d["tph"] * d["ready"], 2)

    return d


# ===================================================
# SAFE INPUTS
# ===================================================
def ask_operator():
    while True:
        v = input("Operator: ").strip()
        if v == "":
            print("❌ Tidak boleh kosong."); continue
        v = fix_name(v)
        if not v.replace(" ", "").isalpha():
            print("❌ Hanya alphabet!"); continue
        return v

def ask_badge_unique(name, ignore=None):
    while True:
        b = input("Badge ID (angka): ").strip()
        if b == "":
            print("❌ Tidak boleh kosong."); continue
        if not b.isdigit():
            print("❌ Badge harus angka!"); continue

        conflict = False
        for i, row in enumerate(fleet):
            if ignore == i:
                continue
            if row["badge"] == b and row["operator"] != name:
                conflict = True; break

        if conflict:
            print("❌ Badge ID sudah dipakai operator lain!")
            continue

        return b

def ask_int(prompt):
    while True:
        v = input(prompt).strip()
        if v == "":
            print("❌ Tidak boleh kosong."); continue
        if not v.isdigit():
            print("❌ Harus bilangan bulat!"); continue
        return int(v)

def ask_float(prompt):
    while True:
        v = input(prompt).strip()
        if v == "":
            print("❌ Tidak boleh kosong."); continue
        v = fix_number(v)
        try:
            return float(v)
        except:
            print("❌ Harus angka menggunakan titik atau koma!")

def ask_hours(prompt, used):
    sisa = SHIFT - used
    while True:
        inp = input(f"{prompt} (sisa {sisa} jam): ").strip()
        if inp == "":
            print("❌ Tidak boleh kosong."); continue
        inp = fix_number(inp)
        try:
            v = float(inp)
        except:
            print("❌ Harus angka!"); continue
        if v > sisa:
            print(f"❌ Total > 12 jam! Sisa hanya {sisa}."); continue
        return v


# ===================================================
# AUTO SKIP
# ===================================================
def must_skip(a, b=0, c=0):
    return (
        a>=12 or b>=12 or c>=12 or
        a+b>=12 or a+c>=12 or b+c>=12 or
        a+b+c>=12
    )


# ===================================================
# SAFE INDEX
# ===================================================
def ask_index(limit, prompt):
    while True:
        v = input(prompt).strip()
        if v == "":
            print("❌ Tidak boleh kosong."); continue
        if not v.isdigit():
            print("❌ Index harus angka!"); continue
        idx = int(v) - 1
        if not (0 <= idx < limit):
            print(f"❌ Index harus 1..{limit}")
            continue
        return idx


# ===================================================
# CREATE
# ===================================================
def create():
    d = {}

    d["operator"] = ask_operator()
    d["badge"] = ask_badge_unique(d["operator"])

    d["delay"] = ask_hours("Delay", 0)
    if must_skip(d["delay"]):
        d.update({"idle":0,"down":0,"distance":0,"speed":0,"inter":0,"wait":0})
        fleet.append(calc(d)); print("⚠ Alat tidak berproduksi"); return

    d["idle"] = ask_hours("Idle", d["delay"])
    if must_skip(d["delay"], d["idle"]):
        d.update({"down":0,"distance":0,"speed":0,"inter":0,"wait":0})
        fleet.append(calc(d)); print("⚠ Alat tidak berproduksi"); return

    d["down"] = ask_hours("Down", d["delay"] + d["idle"])
    if must_skip(d["delay"], d["idle"], d["down"]):
        d.update({"distance":0,"speed":0,"inter":0,"wait":0})
        fleet.append(calc(d)); print("⚠ Alat tidak berproduksi"); return

    d["distance"] = ask_float("Distance (km): ")
    d["speed"] = ask_float("Speed (km/h): ")
    d["inter"] = ask_int("Intersection: ")
    d["wait"] = ask_int("Waiting Equipment: ")

    fleet.append(calc(d))
    print("✅ Created")


# ===================================================
# UPDATE
# ===================================================
def ask_hours_update(prompt, old_val, used):
    """tidak ada yang diubah = enter + cek sisa jam"""
    while True:
        v = input(f"{prompt} ({old_val}): ").strip()
        if v == "":
            return old_val
        try:
            v = float(fix_number(v))
        except:
            print("❌ Harus angka!")
            continue
        sisa = SHIFT - used
        if v > sisa:
            print(f"❌ Total > 12 jam! Sisa hanya {sisa}.")
            continue
        return v


def update():
    show()
    idx = ask_index(len(fleet), "Index update: ")
    old = fleet[idx]
    d = {}

    # -----------------------------
    # OPERATOR
    # -----------------------------
    name = input(f"operator ({old['operator']}): ").strip()
    d["operator"] = fix_name(name) if name else old["operator"]
    if not d["operator"].replace(" ", "").isalpha():
        print("❌ Operator harus alphabet."); return

    # -----------------------------
    # BADGE
    # -----------------------------
    while True:
        b = input(f"badge ({old['badge']}): ").strip()
        if b == "":
            d["badge"] = old["badge"]; break
        if not b.isdigit():
            print("❌ Harus angka!"); continue

        if any(i != idx and row["badge"] == b and row["operator"] != d["operator"]
               for i, row in enumerate(fleet)):
            print("❌ Badge sudah dipakai operator lain!")
            continue

        d["badge"] = b
        break

    # -----------------------------
    # JAM delay, idle, down
    # -----------------------------
    used = 0
    d["delay"] = ask_hours_update("delay", old["delay"], used)
    used = d["delay"]

    d["idle"] = ask_hours_update("idle", old["idle"], used)
    used = d["delay"] + d["idle"]

    d["down"] = ask_hours_update("down", old["down"], used)
    used = d["delay"] + d["idle"] + d["down"]

    # jika total = 12 jam → tidak berproduksi
    if used >= SHIFT:
        d.update({"distance":0,"speed":0,"inter":0,"wait":0})
        fleet[idx] = calc(d)
        print("⚠ Alat tidak berproduksi")
        return

    # -----------------------------
    # Operational
    # -----------------------------
    for k in ["distance", "speed"]:
        v = input(f"{k} ({old[k]}): ").strip()
        d[k] = float(fix_number(v)) if v else old[k]

    for k, label in [("inter","intersection"), ("wait","waiting_equipment")]:
        while True:
            v = input(f"{label} ({old[k]}): ").strip()
            if v == "":
                d[k] = old[k]; break
            if not v.isdigit():
                print("❌ Harus bilangan bulat!"); continue
            d[k] = int(v); break

    fleet[idx] = calc(d)
    print("✅ Updated")

# ===================================================
# DELETE
# ===================================================
def delete():
    show()
    idx = ask_index(len(fleet), "Delete index: ")
    fleet.pop(idx)
    print("✅ Deleted")


# ===================================================
# SHOW
# ===================================================
def show():
    df = pd.DataFrame([calc(dict(x)) for x in fleet])
    df.index = df.index + 1
    df.index.name = "Index"
    display(df)


# ===================================================
# MENU
# ===================================================
print("✅ Program Siap Digunakan")

while True:
    print("""
1. Show Data
2. Create Data
3. Update Data
4. Delete Data
5. Exit
""")

    c = input("Choose: ")

    if c == "1": show()
    elif c == "2": create()
    elif c == "3": update()
    elif c == "4": delete()
    elif c == "5":
        print("✅ Program selesai.")
        break
    else:
        print("❌ Menu tidak valid.")

✅ Program Siap Digunakan

1. Show Data
2. Create Data
3. Update Data
4. Delete Data
5. Exit



Choose:  1


,operator,badge,delay,idle,down,distance,speed,inter,wait,ready,cycle,tphh,trips,tph,prod
Index,,,,,,,,,,,,,,,
1,Shuree,001,0.5,0.0,3,7.0,30,2,1,8.5,53.00,1.13,9.60,271.2,2305.2
2,Sausan,002,1.0,2.1,1,6.7,23,0,1,7.9,51.96,1.15,9.08,276.0,2180.4
3,Ummi,003,3.0,0.0,4,8.0,32,3,0,5.0,54.00,1.11,5.55,266.4,1332.0
4,Rahma,004,3.0,2.5,1,4.5,18,0,0,5.5,42.00,1.43,7.86,343.2,1887.6
5,Billy,005,0.0,0.0,12,0.0,0,0,0,0.0,0.00,0.00,0.00,0.0,0.0



1. Show Data
2. Create Data
3. Update Data
4. Delete Data
5. Exit



KeyboardInterrupt: Interrupted by user